## Similar to mcf_singletiles_decollided.ipynb but using greedy method. --09-06-2026

In [1]:
import argparse
import glob
import gc
import os
import signal
import sys
import time
from collections import defaultdict

import healpy as hp
import numpy as np
from astropy.table import Table
from numpy.random import Generator, PCG64

sys.path.append("/home/zjding/installed_packages/JUST_fiberassign/JUST_fiberassign/") 

In [2]:
from parameters import (
    COLLISION_SEPARATION_ARCSEC,
    COLLISION_SEPARATION_DEG,
    COLLISION_SEPARATION_MM,
    TILE_INNER_RADIUS_DEG,
    TILE_OUTER_RADIUS_DEG,
    R_PATROL,
    R_PATROL_DEG,
)
from utils import _log, get_fiberpos, write_fba_onetile, find_neighboring_fibers, radec2xy, mask_targets_in_tile
from fba_single_tile import (
    _degrade_mtl_priorities_on_disk,
    _fits_path_for_tile,
    _healpix_ids_for_disc,
    fba_onetile,
    load_galaxies_from_mtlpix,
    FBASolveFailed,
)
from greedy_assign import assign_targets_greedy

In [3]:
def _build_greedy_assignment(tile_ra, tile_dec, tile_id, gal_mtl, fiberpos_xy):
    """Single-pass greedy fiber assignment using KDTree and priority ordering."""
    t0 = time.time()

    # Keep only targets inside the tile annulus; radec2xy cannot handle targets
    # beyond the focal-plane interpolation range.
    in_tile = mask_targets_in_tile(
        tile_ra,
        tile_dec,
        gal_mtl["RA"],
        gal_mtl["DEC"],
        TILE_INNER_RADIUS_DEG,
        TILE_OUTER_RADIUS_DEG,
    )
    gal_in_tile = gal_mtl[in_tile]
    n_before = len(np.unique(gal_in_tile["TARGETID"])) if len(gal_in_tile) else 0
    if n_before == 0:
        _log(f"  Tile {tile_id}: no targets in tile annulus")
        return None

    target_pos = np.column_stack(
        radec2xy(tile_ra, tile_dec, gal_in_tile["RA"], gal_in_tile["DEC"])
    )
    target_ids = np.asarray(gal_in_tile["TARGETID"], dtype=np.int64)
    priorities = np.asarray(gal_in_tile["PRIORITY"], dtype=np.float64)
    subpriorities = np.asarray(gal_in_tile["SUBPRIORITY"], dtype=np.float64)

    assigned_ids = assign_targets_greedy(
        fibers_center=fiberpos_xy,
        targets_pos=target_pos,
        targets_ID=target_ids,
        priorities=priorities,
        subpriorities=subpriorities,
        radius=R_PATROL,
        minimum_separation=COLLISION_SEPARATION_MM,
    )

    # Build a fiber -> target_id mapping from the returned order.
    assigned_ids = np.asarray(assigned_ids, dtype=np.int64)
    fiber_ids = np.arange(len(assigned_ids), dtype=np.int32)
    valid = assigned_ids != -1
    assigned_targets_id = assigned_ids[valid].tolist()
    assigned_fiber_id = fiber_ids[valid].tolist()

    # Build a simple reachability dict for write_fba_onetile.
    targets_id_list_alltiles = {f"tile_{tile_id}": {}}
    for fid in fiber_ids:
        targets_id_list_alltiles[f"tile_{tile_id}"][f"fiber_{fid}"] = [int(assigned_ids[fid])]

    _log(
        f"  greedy_assign finished in {time.time() - t0:.2f}s; "
        f"{len(assigned_targets_id)} assignments from {n_before} targets"
    )
    return {
        "assigned_targets_id": assigned_targets_id,
        "assigned_fiber_id": assigned_fiber_id,
        "targets_id_list_alltiles": targets_id_list_alltiles,
    }


def _list_completed_tiles(out_dir):
    completed = set()
    for path in glob.glob(os.path.join(out_dir, "fba_tile_*.fits")):
        base = os.path.basename(path)
        try:
            completed.add(int(base[len("fba_tile_") : -len(".fits")]))
        except ValueError:
            continue
    return completed

In [6]:
input_mockpath="/home/zjding/fiberassignment/JUST/BGS_mock/Junyu_mock/galaxy_cluster/input/lightcone_ra_0_90_dec_0_90_rmagcut20.5_cluster_mask.fits"
input_tilepath="/home/zjding/fiberassignment/JUST/BGS_mock/Junyu_mock/galaxy_cluster/input/tiles_optimized_pass1_2_v0.8.fits"
output_fba_path="./fba/greedy/"
mock_version="v1"
Npasses = 3
ra0=0
ra1=90
dec0=0
dec1=90
n_workers = 2
eval_workers = 1
max_iterations = 1
rand_seed = 100
no_resume = True
nside = 32
# set target priority
priority_initial = 100.0
priority_degraded = 2.0

In [5]:
pool_workers = max(1, n_workers - 2) if n_workers > 2 else n_workers

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

rng = Generator(PCG64(seed=rand_seed))
_log(
    f"rand_seed={rand_seed}, pool_workers={pool_workers}, "
    f"eval_workers={eval_workers}, max_iterations={max_iterations}"
)
_log(f"Patrol radius: {R_PATROL_DEG:.6f} degrees")
_log(
    f"Collision separation: {COLLISION_SEPARATION_ARCSEC} arcsec = "
    f"{COLLISION_SEPARATION_DEG:.6f} deg = {COLLISION_SEPARATION_MM:.3f} mm"
)

fiberpos_xy = get_fiberpos()
N_fibers = fiberpos_xy.shape[0]
_log(f"Number of fibers: {N_fibers}")


input_cat = Table.read(input_mockpath)
mask = (
    (input_cat["ra"] > ra0)
    & (input_cat["ra"] < ra1)
    & (input_cat["dec"] > dec0)
    & (input_cat["dec"] < dec1)
)
input_cat = input_cat[mask]

gal_MTL = Table()

columns = ["TARGETID", "RA", "DEC", "PRIORITY", "SUBPRIORITY"]
for col in columns:
    if col == "TARGETID":
        gal_MTL[col] = input_cat["idx"]
    elif col == "RA":
        gal_MTL[col] = input_cat["ra"]
    elif col == "DEC":
        gal_MTL[col] = input_cat["dec"]
    elif col == "PRIORITY":
        gal_MTL[col] = np.ones(len(input_cat), dtype=float) * priority_initial
        mask = (input_cat["cluster_mask"] == 0)  # distinguish between inside/outside the galaxy cluster center mask
        gal_MTL["PRIORITY"][mask] = priority_degraded
    elif col == "SUBPRIORITY":
        gal_MTL[col] = rng.random(len(input_cat))
    else:
        gal_MTL[col] = input_cat[col]
        
del input_cat
gc.collect()

rand_seed=100, pool_workers=2, eval_workers=1, max_iterations=1
Patrol radius: 0.013021 degrees
Collision separation: 15.625 arcsec = 0.004340 deg = 2.000 mm
Number of fibers: 2184


202

In [7]:
tiles_sub = Table.read(input_tilepath)
##N_tiles = len(tiles_sub)
N_tiles = 11987
# add PASS column to the tiles_sub
tiles_sub["PASS"] = np.array(tiles_sub["TILEID"]//100_000 - 1, dtype=np.int64)

neighboring_fiber_pairs = find_neighboring_fibers(
    fiberpos_xy, patrol_center_separation=12.0
)
_log(f"There are {len(neighboring_fiber_pairs)} paired fibers in one tile")

out_dir = (
    output_fba_path
    + f"/{N_tiles}tiles_{ra0:.1f}ra{ra1:.1f}_{dec0:.1f}dec{dec1:.1f}/seed{rand_seed}/"
)
os.makedirs(out_dir, exist_ok=True)

There are 6312 paired fibers in one tile


In [9]:
nest = False
mtl_dir = out_dir + f"/mtl_nside{nside}/"
mtl_files_exist = bool(glob.glob(os.path.join(mtl_dir, "mtl_healpix_*.fits")))
if no_resume or not mtl_files_exist:
    _log("Split the MTL galaxies into pixels for a fresh fiber-assignment run.")
    npix = hp.nside2npix(nside)
    pix_area_deg2 = hp.nside2pixarea(nside, degrees=True)
    _log(
        f"HEALPix nside={nside}: {npix} pixels, "
        f"~{np.sqrt(pix_area_deg2):.2f} deg/pixel side"
    )

    ra = np.asarray(gal_MTL["RA"], dtype=np.float64)
    dec = np.asarray(gal_MTL["DEC"], dtype=np.float64)
    pix = hp.ang2pix(nside, ra, dec, lonlat=True, nest=nest)

    gal_MTL["HEALPIXID"] = pix.astype(np.int64)
    _log(f"Assigned {len(gal_MTL)} galaxies to {len(np.unique(pix))} non-empty pixels")

    sort_idx = np.argsort(pix, kind="stable")
    pix_sorted = pix[sort_idx]
    unique_pix, start_idx, counts = np.unique(
        pix_sorted, return_index=True, return_counts=True
    )

    pixel_cats = {}
    for pix_id, start, count in zip(unique_pix, start_idx, counts):
        rows = sort_idx[start : start + count]
        pixel_cats[int(pix_id)] = gal_MTL[rows]

    _log(f"Built pixel_cats with {len(pixel_cats)} entries")
    _log(
        "Galaxies per pixel: "
        f"min={counts.min()}, median={np.median(counts):.0f}, max={counts.max()}"
    )

    os.makedirs(mtl_dir, exist_ok=True)
    for pix_id, cat_pix in pixel_cats.items():
        ofile = os.path.join(mtl_dir, f"mtl_healpix_{pix_id:05d}.fits")
        cat_pix.write(ofile, overwrite=True)
    _log(f"Wrote {len(pixel_cats)} pixel catalogs to {mtl_dir}")

    del gal_MTL, pixel_cats
    gc.collect()
else:
    _log(f"Resume: reusing existing MTL pixel files in {mtl_dir}")
    del gal_MTL
    gc.collect()

if no_resume:
    completed_tiles = set()
    _log("Resume disabled (--no_resume); starting from scratch")
else:
    completed_tiles = _list_completed_tiles(out_dir)
    if completed_tiles:
        _log(f"Resume: found {len(completed_tiles)} completed tile(s) in {out_dir}")
    else:
        _log(f"Resume: no completed tiles found in {out_dir}")

near_radius_deg = 1.2 * TILE_OUTER_RADIUS_DEG
_log(
    f"Load the sample from healpixID files, where some galaxies are within "
    f"{near_radius_deg:.6f} deg of the tile center."
)

Split the MTL galaxies into pixels for a fresh fiber-assignment run.
HEALPix nside=32: 12288 pixels, ~1.83 deg/pixel side
Assigned 13032492 galaxies to 1568 non-empty pixels
Built pixel_cats with 1568 entries
Galaxies per pixel: min=3418, median=8422, max=10830
Wrote 1568 pixel catalogs to ./fba/greedy//11987tiles_0.0ra90.0_0.0dec90.0/seed100//mtl_nside32/
Resume disabled (--no_resume); starting from scratch
Load the sample from healpixID files, where some galaxies are within 0.716160 deg of the tile center.


In [10]:
#for passid in range(Npasses):
passid = 1

Ntiles = 20  # number of tiles to run

t0 = time.time()
_log(f"passid: {passid}")
mask = (tiles_sub["PASS"] == passid)
tiles_ra_pass = np.asarray(tiles_sub["RA_NEW"][mask], dtype=np.float64)
tiles_dec_pass = np.asarray(tiles_sub["DEC_NEW"][mask], dtype=np.float64)
tiles_id_pass = np.asarray(tiles_sub["TILEID"][mask], dtype=np.int64)
_log(f"Number of tiles in pass {passid}: {len(tiles_ra_pass)}")

for tile_ra, tile_dec, tile_id in zip(
    tiles_ra_pass[0:Ntiles], tiles_dec_pass[0:Ntiles], tiles_id_pass[0:Ntiles]
):
    out_fits = _fits_path_for_tile(out_dir, tile_id)
    if not no_resume and tile_id in completed_tiles:
        _log(f"Tile {tile_id}: skip (already exists) {out_fits}")
        continue
    if not no_resume and tile_id in skipped_tiles:
        _log(
            f"Tile {tile_id}: skip (assignment previously failed; "
            f"see {skipped_tiles_path})"
        )
        continue

    pix_ids = _healpix_ids_for_disc(
        tile_ra, tile_dec, near_radius_deg, nside, nest=nest
    )
    gal_mtl = load_galaxies_from_mtlpix(
        tile_ra,
        tile_dec,
        mtl_dir,
        near_radius_deg,
        nside=nside,
        nest=nest,
    )
    _log(
        f"Tile {tile_id} ({tile_ra:.4f}, {tile_dec:.4f}): "
        f"loaded {len(gal_mtl)} galaxies from {len(pix_ids)} HEALPix file(s) "
        f"pix={pix_ids.tolist()}"
    )

    t_fba_start = time.time()
    fba_out = _build_greedy_assignment(
        tile_ra,
        tile_dec,
        tile_id,
        gal_mtl,
        fiberpos_xy,
    )
    t_fba_end = time.time()
    _log(f"  fba total time: {t_fba_end - t_fba_start:.2f}s")
    if fba_out is None:
        skipped_tiles.add(int(tile_id))
        _log(f"Tile {tile_id}: assignment skipped\n")
        _log("=====================")
        continue

    assigned_targets_id = fba_out["assigned_targets_id"]
    assigned_fiber_id = fba_out["assigned_fiber_id"]
    targets_id_list_alltiles = fba_out["targets_id_list_alltiles"]
    assigned_tarids_all = list(assigned_targets_id)

    write_fba_onetile(
        tile_id=f"tile_{tile_id}",
        assigned_targets_id=assigned_targets_id,
        assigned_fiber_id=assigned_fiber_id,
        targets_id_list_alltiles=targets_id_list_alltiles,
        out_fits_path=out_fits,
        overwrite=True,
    )
    t_outfits_end = time.time()
    _log(
        f"  Wrote {out_fits} ({len(assigned_targets_id)} assignments); "
        f"takes {t_outfits_end - t_fba_end:.2f}s"
    )

    if assigned_tarids_all:
        affected_pix = np.unique(
            gal_mtl["HEALPIXID"][
                np.isin(gal_mtl["TARGETID"], assigned_tarids_all)
            ]
        )
        n_disk = _degrade_mtl_priorities_on_disk(
            mtl_dir, assigned_tarids_all, priority_degraded, affected_pix
        )
        _log(
            f"  Updated PRIORITY on disk for {n_disk} row(s) "
            f"in {len(affected_pix)} pixel file(s): {affected_pix.tolist()}"
        )
    completed_tiles.add(int(tile_id))
    _log(f"  Update the MTL PRIORITY files in {time.time() - t_outfits_end:.2f}s\n")
    _log("=====================")

_log(f"pass {passid} running time (s): {time.time() - t0:.1f}")

passid: 1
Number of tiles in pass 1: 3996
Tile 208445 (65.4608, 86.6870): loaded 25300 galaxies from 3 HEALPix file(s) pix=[5, 13, 14]
  greedy_assign finished in 2.05s; 1538 assignments from 3165 targets
  fba total time: 2.05s
  Wrote ./fba/greedy//11987tiles_0.0ra90.0_0.0dec90.0/seed100/fba_tile_208445.fits (1538 assignments); takes 0.01s
  Updated PRIORITY on disk for 1538 row(s) in 2 pixel file(s): [5, 14]
  Update the MTL PRIORITY files in 0.03s

Tile 208748 (86.5695, 87.5306): loaded 25930 galaxies from 6 HEALPix file(s) pix=[0, 1, 5, 6, 14, 15]
  greedy_assign finished in 0.62s; 884 assignments from 1628 targets
  fba total time: 0.62s
  Wrote ./fba/greedy//11987tiles_0.0ra90.0_0.0dec90.0/seed100/fba_tile_208748.fits (884 assignments); takes 0.01s
  Updated PRIORITY on disk for 884 row(s) in 2 pixel file(s): [5, 14]
  Update the MTL PRIORITY files in 0.04s

Tile 208540 (45.6933, 83.7342): loaded 32180 galaxies from 4 HEALPix file(s) pix=[13, 25, 26, 42]
  greedy_assign finished